# 01_mock: Progressive Banking Tool-Use Agent (Stage-Gated)

Timebox: **55 minutes**  
Language: **Python (Colab)**

This mock combines:
1. Tool-use agent loop implementation.
2. Prompt + schema shaping.
3. Progressive complexity in CodeSignal-style levels.

## Scenario
Build a support agent for a small banking workflow. The model can call local tools for balances, transfers, and audits.

## Stage-gated levels
- **Level 1:** Prompt + schema + single tool call loop.
- **Level 2:** Multiple tool calls in one turn with strict sequencing.
- **Level 3:** Business-logic mutation + safe error handling.
- **Level 4:** `pause_turn` continuation + bounded retry + max-steps safety.

## What to implement
1. `build_system_prompt`
2. `build_tool_schemas`
3. `execute_tool_call`
4. `run_agent`

## Time guidance
- 10 min: read staged tests and lock contracts.
- 35 min: implement minimal passing behavior for all levels.
- 10 min: harden edge cases and re-run stage sequence.


In [ ]:
import inspect
import json
from copy import deepcopy
from typing import Any, Callable

ACCOUNTS = {
    "a-100": {"account_id": "a-100", "balance": 300.0},
    "a-200": {"account_id": "a-200", "balance": 120.0},
}

LEDGER = {
    "a-100": [
        {"kind": "deposit", "amount": 300.0},
    ],
    "a-200": [
        {"kind": "deposit", "amount": 120.0},
    ],
}

TRANSFERS: list[dict[str, Any]] = []
AUDIT_ATTEMPTS: dict[str, int] = {}


def reset_state() -> None:
    ACCOUNTS["a-100"]["balance"] = 300.0
    ACCOUNTS["a-200"]["balance"] = 120.0
    LEDGER["a-100"] = [{"kind": "deposit", "amount": 300.0}]
    LEDGER["a-200"] = [{"kind": "deposit", "amount": 120.0}]
    TRANSFERS.clear()
    AUDIT_ATTEMPTS.clear()


def get_balance(account_id: str) -> dict[str, Any]:
    if account_id not in ACCOUNTS:
        raise ValueError("unknown_account")
    return deepcopy(ACCOUNTS[account_id])


def list_recent_transactions(account_id: str, limit: int = 3) -> list[dict[str, Any]]:
    if account_id not in LEDGER:
        raise ValueError("unknown_account")
    return deepcopy(LEDGER[account_id][-limit:])


def transfer_funds(from_account: str, to_account: str, amount: float) -> dict[str, Any]:
    if from_account not in ACCOUNTS or to_account not in ACCOUNTS:
        raise ValueError("unknown_account")
    if amount <= 0:
        raise ValueError("amount_must_be_positive")
    if ACCOUNTS[from_account]["balance"] < amount:
        raise ValueError("insufficient_funds")

    ACCOUNTS[from_account]["balance"] -= amount
    ACCOUNTS[to_account]["balance"] += amount
    out = {"from": from_account, "to": to_account, "amount": amount, "status": "posted"}

    LEDGER[from_account].append({"kind": "transfer_out", "amount": amount, "to": to_account})
    LEDGER[to_account].append({"kind": "transfer_in", "amount": amount, "from": from_account})
    TRANSFERS.append(deepcopy(out))
    return out


def run_account_audit(account_id: str) -> dict[str, Any]:
    count = AUDIT_ATTEMPTS.get(account_id, 0)
    AUDIT_ATTEMPTS[account_id] = count + 1
    if count == 0:
        raise RuntimeError("transient_audit_backend_timeout")
    return {"account_id": account_id, "status": "audit_ok"}


TOOL_REGISTRY: dict[str, Callable[..., Any]] = {
    "get_balance": get_balance,
    "list_recent_transactions": list_recent_transactions,
    "transfer_funds": transfer_funds,
    "run_account_audit": run_account_audit,
}


class ScriptedModel:
    def __init__(self, responses: list[dict[str, Any]]) -> None:
        self._responses = deepcopy(responses)
        self._index = 0
        self.last_system_prompt: str | None = None
        self.last_tools: list[dict[str, Any]] = []

    def __call__(self, messages: list[dict[str, Any]], system_prompt: str, tools: list[dict[str, Any]]) -> dict[str, Any]:
        self.last_system_prompt = system_prompt
        self.last_tools = deepcopy(tools)
        if self._index >= len(self._responses):
            return {"stop_reason": "end_turn", "content": [{"type": "text", "text": "no scripted response"}]}
        response = self._responses[self._index]
        self._index += 1
        return deepcopy(response)


In [ ]:
def build_system_prompt() -> str:
    """Return concise instructions for a safe tool-using support agent."""
    # TODO:
    # - Instruct the model to use tools before making account-specific claims.
    # - Instruct the model to avoid fabricated tool outputs.
    # - Instruct the model to ask clarifying questions for missing transfer fields.
    raise NotImplementedError


def build_tool_schemas(tool_registry: dict[str, Callable[..., Any]]) -> list[dict[str, Any]]:
    """Build minimal tool definitions with required args from function signatures."""
    # TODO: return list of {name, description, input_schema}
    raise NotImplementedError


def execute_tool_call(
    tool_call: dict[str, Any],
    tool_registry: dict[str, Callable[..., Any]],
    cache: dict[str, dict[str, Any]],
) -> dict[str, Any]:
    """Execute one tool call with deterministic cache key and one retry on RuntimeError."""
    # TODO:
    # - Validate required tool_call fields and arg presence.
    # - Cache by tool_name + stable-json(input).
    # - Retry once only for RuntimeError.
    # - Return {type, tool_use_id, is_error, from_cache, content}.
    raise NotImplementedError


def run_agent(
    user_prompt: str,
    model: Callable[[list[dict[str, Any]], str, list[dict[str, Any]]], dict[str, Any]],
    tool_registry: dict[str, Callable[..., Any]],
    max_steps: int = 8,
) -> dict[str, Any]:
    """Run loop until end_turn with tool_use, pause_turn, and max-step safety."""
    # TODO:
    # - Initialize message list in Messages-API shape.
    # - Pass system prompt and tool schemas on every model call.
    # - On tool_use: execute all tool calls and append one user tool_result message.
    # - On pause_turn: continue without adding a new user message.
    # - On end_turn: return {final_text, messages, stats}.
    # - Raise RuntimeError("max_steps_exceeded") when out of steps.
    raise NotImplementedError


## Run Tests
Run the stage-gated test cell after implementing all TODO sections.


In [ ]:
# Chunk overview: stage-gated checks that emulate progressive complexity.

def _tool_result_blocks(messages: list[dict[str, Any]]) -> list[dict[str, Any]]:
    blocks: list[dict[str, Any]] = []
    for m in messages:
        if m.get("role") != "user":
            continue
        for b in m.get("content", []):
            if b.get("type") == "tool_result":
                blocks.append(b)
    return blocks


def run_level_1() -> None:
    reset_state()
    prompt = build_system_prompt().lower()
    assert "use tools" in prompt
    assert "never fabricate" in prompt

    defs = build_tool_schemas(TOOL_REGISTRY)
    assert sorted(d["name"] for d in defs) == sorted(TOOL_REGISTRY.keys())

    model = ScriptedModel(
        [
            {
                "stop_reason": "tool_use",
                "content": [
                    {"type": "tool_use", "id": "l1-1", "name": "get_balance", "input": {"account_id": "a-100"}}
                ],
            },
            {"stop_reason": "end_turn", "content": [{"type": "text", "text": "Balance is 300."}]},
        ]
    )
    result = run_agent("check balance", model, TOOL_REGISTRY)
    assert "300" in result["final_text"]


def run_level_2() -> None:
    reset_state()
    model = ScriptedModel(
        [
            {
                "stop_reason": "tool_use",
                "content": [
                    {"type": "tool_use", "id": "l2-1", "name": "list_recent_transactions", "input": {"account_id": "a-100", "limit": 1}},
                    {"type": "tool_use", "id": "l2-2", "name": "get_balance", "input": {"account_id": "a-100"}},
                ],
            },
            {"stop_reason": "end_turn", "content": [{"type": "text", "text": "latest tx + balance ready"}]},
        ]
    )
    result = run_agent("details", model, TOOL_REGISTRY)
    blocks = _tool_result_blocks(result["messages"])
    assert len(blocks) == 2

    assistant_idx = next(i for i, m in enumerate(result["messages"]) if m["role"] == "assistant")
    assert result["messages"][assistant_idx + 1]["role"] == "user"
    assert all(b.get("type") == "tool_result" for b in result["messages"][assistant_idx + 1]["content"])


def run_level_3() -> None:
    reset_state()
    model = ScriptedModel(
        [
            {
                "stop_reason": "tool_use",
                "content": [
                    {"type": "tool_use", "id": "l3-1", "name": "transfer_funds", "input": {"from_account": "a-100", "to_account": "a-200", "amount": 25.0}},
                    {"type": "tool_use", "id": "l3-2", "name": "not_a_tool", "input": {"x": 1}},
                ],
            },
            {"stop_reason": "end_turn", "content": [{"type": "text", "text": "transfer attempted"}]},
        ]
    )
    result = run_agent("transfer", model, TOOL_REGISTRY)

    assert ACCOUNTS["a-100"]["balance"] == 275.0
    assert ACCOUNTS["a-200"]["balance"] == 145.0

    blocks = _tool_result_blocks(result["messages"])
    bad = [b for b in blocks if b["tool_use_id"] == "l3-2"][0]
    assert bad["is_error"] is True
    assert "unknown_tool" in bad["content"]


def run_level_4() -> None:
    reset_state()
    model = ScriptedModel(
        [
            {"stop_reason": "pause_turn", "content": [{"type": "text", "text": "processing"}]},
            {
                "stop_reason": "tool_use",
                "content": [
                    {"type": "tool_use", "id": "l4-1", "name": "run_account_audit", "input": {"account_id": "a-100"}}
                ],
            },
            {"stop_reason": "end_turn", "content": [{"type": "text", "text": "audit complete"}]},
        ]
    )
    result = run_agent("audit", model, TOOL_REGISTRY)
    assert "audit complete" in result["final_text"]
    assert AUDIT_ATTEMPTS["a-100"] == 2

    loop_model = ScriptedModel(
        [
            {"stop_reason": "tool_use", "content": [{"type": "tool_use", "id": "mx-1", "name": "get_balance", "input": {"account_id": "a-100"}}]},
            {"stop_reason": "tool_use", "content": [{"type": "tool_use", "id": "mx-2", "name": "get_balance", "input": {"account_id": "a-100"}}]},
        ]
    )
    try:
        run_agent("loop", loop_model, TOOL_REGISTRY, max_steps=1)
        raise AssertionError("Expected max_steps_exceeded")
    except RuntimeError as exc:
        assert "max_steps_exceeded" in str(exc)


def run_exam01_tests() -> None:
    run_level_1()
    print("Level 1 passed")
    run_level_2()
    print("Level 2 passed")
    run_level_3()
    print("Level 3 passed")
    run_level_4()
    print("Level 4 passed")
    print("01_mock tests passed")


run_exam01_tests()
